In [1]:
import pandas as pd
import os
import warnings
from pandasql import sqldf
from datetime import datetime, timedelta
import glob
warnings.filterwarnings("ignore")
import openpyxl

### Nielsen

In [2]:
dir = os.getcwd()
# Important!! Make sure the file exist and refreshed first
my_ldb = pd.read_excel(f'{dir}/../Data Source/Nielsen/Nielsen O+O May26_230626.xlsx', sheet_name='MY Nielsen Mass Medic')
my_ldb.head()

,Currency,Markets,Periods,GENDER,LOREAL_BRANDTYPE,MANUFACTURER,BRAND,LOREALFUNCTION,Sales Value,Sales Units
0,MYR,Pen Malaysia + EM Modern Trade,Apr 23 - w/e 30/04/23,WOMAN,MEDIC,TOTAL PRIVATE LABEL & EXCLUSIVE BRANDS,NaN,NaN,641332.194,31373.393
1,MYR,Pen Malaysia + EM Modern Trade,Apr 23 - w/e 30/04/23,WOMAN,MEDIC,NaN,AVENE,ANTI ACNE,111059.980,2017.992
2,MYR,Pen Malaysia + EM Modern Trade,Apr 23 - w/e 30/04/23,WOMAN,MEDIC,NaN,AVENE,ANTI AGE,53024.951,657.424
3,MYR,Pen Malaysia + EM Modern Trade,Apr 23 - w/e 30/04/23,WOMAN,MEDIC,NaN,AVENE,BASIC,60673.050,1105.423
4,MYR,Pen Malaysia + EM Modern Trade,Apr 23 - w/e 30/04/23,WOMAN,MEDIC,NaN,AVENE,HYDRATION,728009.026,14174.802


In [3]:
# Formatting months
def month_to_number(month):
    months = {'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04', 'May': '05', 'Jun': '06', 'Jul': '07', 'Aug': '08', 'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12'}
    return months.get(month, month)

In [4]:
# Important!! Brand Mapping - Any new Brands need to be added here - LDB Brands - Make sure consistent for all O+O Scripts
mass_medic = [
    'ACNE AID', 'ACNES', 'AVEENO', 'BALNEUM', 'BENZAC', 'BIO-OIL', 'CARMEX', 'CERAVE', 'CETAPHIL', 'CUREL',
    'DERMATIX', 'DERMAVEEN', 'DIFFERIN', 'DR.G', 'DR.YU', 'EGO', 'EUBOS', 'LACTACYD', 'LINOLA', 'MUSTELA',
    'NEUTROGENA', 'PANOXYL', 'PHYSIOGEL', 'SEBAMED', 'TOPICREM', 'VANICREAM', 'WIS', 'XHEKPON', 'FIRST AID BEAUTY',
    'AQUAPHOR', 'NOBACTER', 'LUBRIDERM', 'NEOSPORIN', 'DARROW', 'DEXERYL', 'ALERGIBON', 'ALPHYGIENE', 'BABIGOZ',
    'CANDERMYL', 'GALDERMA', 'GALDERMA OTHER', 'HELIOBLOC', 'HYDRODERM OMEGA', 'IOCON', 'IONIL', 'MACROLANE',
    'MICROBAN', 'MICROSUN', 'NESTLE', 'NUTRASPA', 'OBSERVANCE', 'PHYGIENE', 'R-GEN', 'SENTIAL', 'ACHE', 'ACNAID',
    'ACNE FREE', 'ACOFAR', 'ADDAX', 'AKILDIA', 'ALBOLENE', 'AMLACTIN', 'ANSEBIC', 'AQUA SOAP', 'AQUA-SOAP',
    'AVITIL', 'AZULENNE', 'BACCIDE', 'BEAUTY PLUS', 'BEDOOK', 'BEPANTHEN/BEPANTHOL', 'BETAGRANULOS', 'BIAFINE',
    'BIOBLAS', 'BIOCLIN', 'BIOLIQ', 'BIOXCIN', 'BLUE LIZARD', 'BODYSOL', 'BONAVEN', 'BOROLINE', 'CERAMOL',
    'CERTAIN DRI', 'CETOPIC', 'CHICCO', 'CICAMEL', 'COOPER', 'COTARYL', 'CRISTALIA', 'DECUBAL', 'DERMAC',
    'DERMACTIVE', 'DERMADRATE', 'DERMAGE', 'DERMAKERI', 'DERMENA', 'DERMON', 'DERSUPRIL', 'DEUMAVAN',
    'DOCTISSIMO PARAPHARMACIE', 'DR.LI', 'DR.LIDERMO', 'DRAYEX', 'DX2', 'E45', 'ELDOPAQUE', 'EMOLIENTA', 'EMOLIN',
    'EMOLIUM', 'EPIMAX', 'EVASOL', 'FARMOQUIMICA', 'FILTROSOL', 'FLUOCIN', 'FREI OEL (BOUHON)', 'GALENCO', 'GIFRER',
    'GILBERT', 'GOLD BOND', 'HAMILTON', 'HIDRAFIL', 'HIPOSOL', 'HYALIX', 'IDROVEL', 'IHADA', 'INFASIL',
    'INTERAPOTHEK', 'IRALTONE', 'ITANIDERM', 'KAMILODERM', 'KETOXIN', 'KINERASE', 'KORA', 'LACTIBON',
    'LACTO CALAMINE', 'LETI', 'LIFAR', 'LIPODERM', 'LOTRIMIN', 'MARQUE VERTE', 'MICRORET', 'MITOSYL', 'MODERM',
    'MULTIDERMOL', 'MUSSVITAL', 'NEUTRA LICE', 'NEUTRAPHARM', 'NORDIN', 'NUMIS', 'NUMIS MED', 'NURAPHARM',
    'NUTREM', 'NUTRISIL', 'OILATUM', 'OILLAN', 'OSMIN', 'OTC IBERICA', 'PANVEL DERMATIV', 'PARABOTICA',
    'PHARMACTIV', 'PHARMASEPT', 'PHISOHEX', 'PROCICAR', 'REGENERUM', 'RESTIV', 'RESTIVOIL', 'REVALESKIN', 'ROCHE',
    'ROGE CAVAILLES', 'ROYALCARE', 'RUGARD (SCHEFFLER)', 'SALILEX', 'SALLVE', 'SARNA', 'SAUGELLA', 'SEBORADIN',
    'SHADE', 'SMOOTH-E', 'SOLAR FOAM', 'S-OLE', 'SPECTRABAN', 'STANHOME FAMILY EXPERT', 'STIEFEL', 'STIEPROX',
    'STIPROX', 'STIPROXAL', 'TARMED', 'TRACTOPON', 'TRI DERMA MD', 'UREADERM', 'UVEIL-PS', 'UVESOL', 'VEA',
    'VENUSIA', 'VITA CITRAL', 'VITALIFE', 'ZODIAC','QV', 'BOBAI',
    'COLLAGE','DERMAREST','EPIZONE E','GLAMY LAB','LU MILD','NOLAVER','OXECURE','RIUP','SEBCUR','SELENGENA','SEROPIPE','SHAAN','STAR VILLE','STRONGVILLE','SYNOBAR','UREMOL',
    'URISEC','ZINPLEX'
]

df = pd.read_excel('Mapping.xlsx', sheet_name='Medic')
non_mass_medic = df.iloc[:, 0].dropna().astype(str).tolist()

# Add manually defined list
non_mass_medic1 = [
    'EUCERIN', 'LA ROCHE POSAY', 'VICHY', 'AVENE', 'DR MORITA', 'HIRUSCAR', 'URIAGE', 'SKINCEUTICALS', 'DECLEOR',
    'SANOFLORE', 'AQUAPHOR', 'BIODERMA', 'ISDIN', 'LIERAC', 'FILORGA', 'PROACTIV', 'ROC', 'WINONA', 'DR CILABO',
    'EMOLIUM', 'PHARMACERIS', 'CAUDALIE', 'NUXE', 'RODAN', 'FIELDS', 'LIBREDERM', 'MANTECORP', 'DR. CI : LABO'
]


# Combine both lists, remove duplicates, and standardize casing (if needed)
combined_non_mass_medic = list(set(non_mass_medic + non_mass_medic1))

# Optional: If you want to preserve order (Excel first, then manual additions)
combined_non_mass_medic = list(dict.fromkeys(non_mass_medic + non_mass_medic1))

print('Total mass_medic Brands: ', len(mass_medic))
print('Total non_mass_medic Brands: ', len(combined_non_mass_medic))

Total mass_medic Brands:  217
Total non_mass_medic Brands:  1133


In [5]:
# Grouping Brands into Mass Medic and Non Mass Medic
def mass_medic_group(row):
    if row['BRAND'] in mass_medic:
        return 'Mass Medical'
    else:
        return 'Non-Mass Medical'

In [6]:
# Important!! Try to understand the filter and logic here
my_ldb['On/Offline'] = 'Offline'
my_ldb['Year'] = my_ldb['Periods'].str.extract('(\d+)', expand=False).astype(int) + 2000
my_ldb['Month Name'] = my_ldb['Periods'].str.split().str[0]
my_ldb['Period'] = my_ldb['Month Name'].apply(month_to_number)
my_ldb['Platform'] = 'Nielsen'
my_ldb['Category'] = 'SKINCARE'
my_ldb['Sub-Category'] = my_ldb['LOREALFUNCTION']
my_ldb['Mass/non-mass (subdivision)'] = my_ldb.apply(mass_medic_group, axis=1)
my_ldb['Brand'] = my_ldb['BRAND']
my_ldb['Market'] = 'MEDIC MARKET'
my_ldb['Country'] = 'MY'

In [7]:
# Filter and aggregate data for Brand and Market
ldb_brand = my_ldb[my_ldb['BRAND'].isin(mass_medic) | my_ldb['BRAND'].isin(combined_non_mass_medic)].groupby(['Country','On/Offline', 'Year', 'Platform', 'Category', 'Sub-Category', 'Mass/non-mass (subdivision)', 'Brand', 'Period'])[['Sales Value','Sales Units']].sum().reset_index()
ldb_market = my_ldb.groupby(['Country','On/Offline', 'Year', 'Platform', 'Category', 'Mass/non-mass (subdivision)', 'Market', 'Period'])[['Sales Value','Sales Units']].sum().reset_index()
# Market Sub-Category should be empty
ldb_market['Sub-Category'] = ''
# Rename Market to Brand
ldb_market = ldb_market.rename(columns={'Market': 'Brand'})
ldb_market.tail(3)

,Country,On/Offline,Year,Platform,Category,Mass/non-mass (subdivision),Brand,Period,Sales Value,Sales Units,Sub-Category
73,MY,Offline,2026,Nielsen,SKINCARE,Non-Mass Medical,MEDIC MARKET,03,1.411191e+07,245394.041,
74,MY,Offline,2026,Nielsen,SKINCARE,Non-Mass Medical,MEDIC MARKET,04,1.268418e+07,189409.651,
75,MY,Offline,2026,Nielsen,SKINCARE,Non-Mass Medical,MEDIC MARKET,05,1.202019e+07,189893.507,


In [8]:
# Merge Brand and Market
my_ldb_offline = pd.concat([ldb_market, ldb_brand], ignore_index=True)
my_ldb_offline.head(3)

,Country,On/Offline,Year,Platform,Category,Mass/non-mass (subdivision),Brand,Period,Sales Value,Sales Units,Sub-Category
0,MY,Offline,2023,Nielsen,SKINCARE,Mass Medical,MEDIC MARKET,04,4253813.919,122416.178,
1,MY,Offline,2023,Nielsen,SKINCARE,Mass Medical,MEDIC MARKET,05,3808601.820,101573.747,
2,MY,Offline,2023,Nielsen,SKINCARE,Mass Medical,MEDIC MARKET,06,3955823.212,102667.598,


### OMT LDB

In [9]:
# Important!! Make sure the file exists and is updated first
files = glob.glob(f'{dir}/../Data Source/OMT - O+O/MY LDB/*.xlsx')

# List to store DataFrames
dfs = []

# Read each file and append to the list
for file in files:
    print(f'Loading file: {file}')  # Print the name of the file being loaded
    df = pd.read_excel(file)
    dfs.append(df)

# Concatenate all DataFrames into one
ldb_data = pd.concat(dfs, ignore_index=True)


Loading file: c:\Users\balatarsini_avinitya\Downloads\CPD & LDB O+O - May\loreal-report-automation (2)\O+O/../Data Source/OMT - O+O/MY LDB\OMT MY LDB 2024-01.xlsx
Loading file: c:\Users\balatarsini_avinitya\Downloads\CPD & LDB O+O - May\loreal-report-automation (2)\O+O/../Data Source/OMT - O+O/MY LDB\OMT MY LDB 2024-02.xlsx
Loading file: c:\Users\balatarsini_avinitya\Downloads\CPD & LDB O+O - May\loreal-report-automation (2)\O+O/../Data Source/OMT - O+O/MY LDB\OMT MY LDB 2024-03.xlsx
Loading file: c:\Users\balatarsini_avinitya\Downloads\CPD & LDB O+O - May\loreal-report-automation (2)\O+O/../Data Source/OMT - O+O/MY LDB\OMT MY LDB 2024-04.xlsx
Loading file: c:\Users\balatarsini_avinitya\Downloads\CPD & LDB O+O - May\loreal-report-automation (2)\O+O/../Data Source/OMT - O+O/MY LDB\OMT MY LDB 2024-05.xlsx
Loading file: c:\Users\balatarsini_avinitya\Downloads\CPD & LDB O+O - May\loreal-report-automation (2)\O+O/../Data Source/OMT - O+O/MY LDB\OMT MY LDB 2024-06.xlsx
Loading file: c:\Users

In [10]:
# Rename platform
def map_platform(row):
    if row['Mall Type'] == 'Shopee Mall' :
        return 'Shopee Mall'
    elif row['Mall Type'] == 'Lazada Mall' :
        return 'Lazada Mall'
    elif row['Mall Type'] == 'Tiktok Mall':
        return 'Tik Tok'
    else:
        return 'Others'

In [11]:
# Grouping Brands into Mass Medic and Non Mass Medic
def map_division(row):
    if row['Brand'] in mass_medic:
        return 'Mass Medical'
    else:
        return 'Non-Mass Medical'

In [12]:
# Rename Columns for consistency
ldb_data = ldb_data.rename(columns={'Total Est. Sales Local': 'Sales Value', "Loreal 1P Est Sales Local": 'Loreal Sales Value', 'Total units sold': 'Sales Units', 'Category L2':'Category_x'})

In [13]:
# Important!! Try to understand the filter and logic here
ldb_data = ldb_data[ldb_data['Category L1'] == 'SKIN CARE']
# Keep only specified mall types
ldb_data = ldb_data[ldb_data['Mall Type'].isin(['Shopee Mall', 'Lazada Mall','Tiktok Mall'])]
# For SUN CARE, only include rows where Category L3 is 'FACE PROTECTION'
# otherwise keep other sub-categories as-is
ldb_data = ldb_data[(ldb_data['Category_x'] != 'SUN CARE') | ((ldb_data['Category_x'] == 'SUN CARE') & (ldb_data['Category L3'] == 'FACE PROTECTION'))]
ldb_data['On/Offline'] = 'Online'
ldb_data[['Year', 'Period']] = ldb_data['Year Month'].str.split('-', expand=True)
ldb_data['Platform'] = ldb_data.apply(map_platform, axis=1)
ldb_data['Category'] = 'SKINCARE'
ldb_data['Sub-Category'] = ldb_data['Category_x']
ldb_data['Mass/non-mass (subdivision)'] = ldb_data.apply(map_division, axis=1)
ldb_data['Market'] = 'MEDIC MARKET'
ldb_data.info()

<class 'pandas.DataFrame'>
Index: 128672 entries, 0 to 143358
Data columns (total 22 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Country                      128672 non-null  str    
 1   Year Month                   128672 non-null  str    
 2   Universe                     128672 non-null  str    
 3   Brand                        128672 non-null  str    
 4   Category L1                  128672 non-null  str    
 5   Category_x                   128672 non-null  str    
 6   Mall Type                    128672 non-null  str    
 7   Category L3                  128672 non-null  str    
 8   Product                      128672 non-null  str    
 9   Benefits                     128672 non-null  str    
 10  Formats                      128672 non-null  str    
 11  Sales Value                  128672 non-null  float64
 12  Loreal Sales Value           5224 non-null    float64
 13  Sales Units    

In [14]:
# Filter and aggregate data for Brand and Market
ldb_market = ldb_data.groupby(['On/Offline', 'Year', 'Platform', 'Category', 'Sub-Category', 'Mass/non-mass (subdivision)', 'Market', 'Period'])[['Sales Value','Sales Units','Loreal Sales Value']].sum().reset_index()
ldb_market = ldb_market.rename(columns={'Market': 'Brand'})
# ldb_market = ldb_market[~((ldb_market['Sub-Category'] == 'Body Care') & (ldb_market['Mass/non-mass (subdivision)'] == 'Non-Mass Medical'))]
ldb_brand = ldb_data.groupby(['On/Offline', 'Year', 'Platform', 'Category', 'Sub-Category', 'Mass/non-mass (subdivision)', 'Brand', 'Period'])[['Sales Value','Sales Units','Loreal Sales Value']].sum().reset_index()
# ldb_brand = ldb_brand[~(ldb_brand['Sub-Category'] == 'Body Care')]
ldb_brand.tail(3)

,On/Offline,Year,Platform,Category,Sub-Category,Mass/non-mass (subdivision),Brand,Period,Sales Value,Sales Units,Loreal Sales Value
7388,Online,2026,Tik Tok,SKINCARE,SUN CARE,Non-Mass Medical,SOME BY MI,04,9679.63,217.0,0.0
7389,Online,2026,Tik Tok,SKINCARE,SUN CARE,Non-Mass Medical,SOME BY MI,05,9158.51,229.0,0.0
7390,Online,2026,Tik Tok,SKINCARE,SUN CARE,Non-Mass Medical,URIAGE,02,99.11,1.0,0.0


In [15]:
# Consolidate Brand and Market to one dataframe
my_ldb_online = pd.concat([ldb_market, ldb_brand], ignore_index=True)

### Final Transformation Based on Srishti WF

In [16]:
# Merge Offline and Online Data 
# my_ldb_final = pd.concat([my_ldb_offline], ignore_index=True)
my_ldb_final = pd.concat([my_ldb_offline, my_ldb_online], ignore_index=True)
my_ldb_final


,Country,On/Offline,Year,Platform,Category,Mass/non-mass (subdivision),Brand,Period,Sales Value,Sales Units,Sub-Category,Loreal Sales Value
0,MY,Offline,2023,Nielsen,SKINCARE,Mass Medical,MEDIC MARKET,04,4253813.919,122416.178,,NaN
1,MY,Offline,2023,Nielsen,SKINCARE,Mass Medical,MEDIC MARKET,05,3808601.820,101573.747,,NaN
2,MY,Offline,2023,Nielsen,SKINCARE,Mass Medical,MEDIC MARKET,06,3955823.212,102667.598,,NaN
3,MY,Offline,2023,Nielsen,SKINCARE,Mass Medical,MEDIC MARKET,07,4036084.694,104910.278,,NaN
4,MY,Offline,2023,Nielsen,SKINCARE,Mass Medical,MEDIC MARKET,08,4076248.712,103513.265,,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
9907,NaN,Online,2026,Tik Tok,SKINCARE,Non-Mass Medical,SOME BY MI,02,7471.300,168.000,SUN CARE,0.0
9908,NaN,Online,2026,Tik Tok,SKINCARE,Non-Mass Medical,SOME BY MI,03,10164.080,230.000,SUN CARE,0.0
9909,NaN,Online,2026,Tik Tok,SKINCARE,Non-Mass Medical,SOME BY MI,04,9679.630,217.000,SUN CARE,0.0
9910,NaN,Online,2026,Tik Tok,SKINCARE,Non-Mass Medical,SOME BY MI,05,9158.510,229.000,SUN CARE,0.0


In [17]:
# Important!! List of Brand to include in the report - New Brands need to be added here too
brand_list = ['MEDIC MARKET', 'CETAPHIL', 'QV', 'SEBAMED', 'PHYSIOGEL', 'AVENE', 'DR MORITA', 'HIRUSCAR', 'URIAGE', 'EUCERIN', 'LA ROCHE POSAY', 'NEUTROGENA', 'VICHY', 'CUREL', 'EGO QV', 'BIODERMA', 'SKINCEUTICALS', 'DECLEOR', 'SANOFLORE', 'AQUAPHOR', 'ISDIN', 'LIERAC', 'FILORGA', 'PROACTIV', 'ROC', 'WINONA', 'DR CILABO', 'EMOLIUM0', 'PHARMACERIS', 'CAUDALIE', 'NUXE', 'RODAN', 'FIELDS', 'LIBREDERM', 'MANTECORP', 'DR. CI : LABO','CERAVE','DR.G']
print('Total brand_list Brands: ', len(brand_list))

Total brand_list Brands:  38


In [18]:
# Rename Brand
def map_brand(row):
    if row['Brand'] == 'Avène':
        return 'AVENE'
    elif row['Brand'] == 'EGO QV':
        return 'QV'
    elif row['Brand'] == 'SKIN CEUTICALS':
        return 'SKINCEUTICALS'
    elif row['Brand'] == 'LA ROSÈE':
        return 'LA ROSEE'
    else:
        return row['Brand']

In [19]:
# Group Brand into loreal brand and total
def map_group(row):
    if row['Brand'] in ['LA ROCHE POSAY', 'VICHY', 'SKINCEUTICALS', 'CERAVE','DR.G']:
        return "L'Oreal"
    elif row['Brand'] == 'MEDIC MARKET':
        return 'Total'
    else:
        return 'Other'

In [20]:
# Category Mapping
def map_category(row):
    if row['Category'] == 'HAIR':
        return 'Haircare'
    elif row['Category'] == 'MAKE UP':
        return 'Makeup'
    elif (row['Category'] == 'SKINCARE') & (row['Sub-Category'] == 'BODY CARE'):
        return 'Bodycare'
    elif (row['Category'] == 'SKINCARE') & (row['Sub-Category'] == 'SUN CARE'):
        return 'Suncare'
    elif (row['Category'] == 'SKINCARE'):
        return 'Facecare'
    else:
        return 'Other'

In [21]:
# Use Loreal Sales Value for Loreal Brand
import math

def map_sellout(row):
    if pd.isna(row['Sales Value']):  # Check if the value is NaN
        return 0  # Return 0 if NaN
    elif isinstance(row['Sales Value'], int):  # Check if the value is already an integer
        return row['Sales Value']
    elif (row["L'Oreal/Other"] == "L'Oreal") and (row['On/Offline'] == 'Online'):
        return row['Loreal Sales Value']
    else:
        try:
            return math.ceil(row['Sales Value'])  # Round up to the nearest integer
        except TypeError:
            return 0  # Handle any other exceptions by returning 0

In [22]:
# Important!! Try to understand the filter and logic here
my_ldb_final = my_ldb_final[my_ldb_final['Brand'].isin(brand_list)]
my_ldb_final['O+O Brand'] = my_ldb_final.apply(map_brand, axis=1)
my_ldb_final["L'Oreal/Other"] = my_ldb_final.apply(map_group, axis=1)
my_ldb_final['O+O Category'] = my_ldb_final.apply(map_category, axis=1)
my_ldb_final['Sellout'] = my_ldb_final.apply(map_sellout, axis=1)

# Remove leading zeros and convert to integer
my_ldb_final['Period'] = my_ldb_final['Period'].astype(str).str.lstrip('0').astype(int)

In [23]:
# Sort Columns
# order = ['On/Offline', 'Year', 'Platform', 'Category', 'Sub-Category', 'Mass/non-mass (subdivision)', 'Brand', 'Period', 'Sales Value', 'Sales Units', 'Loreal Sales Value']
order = ['On/Offline', 'Year', 'Platform', 'Category', 'Sub-Category', 'Mass/non-mass (subdivision)', 'Brand', 'Period', 'Sales Value', 'Sales Units']
my_ldb_final = my_ldb_final[order]
# Rename Columns
# my_ldb_final.rename(columns={'Loreal Sales Value': 'ACD 1P'}, inplace=True)

In [24]:
# Year string to int
my_ldb_final ['Year'] = my_ldb_final ['Year'].astype(int)
my_ldb_final = my_ldb_final[my_ldb_final['Year'] >= 2021]
# Aggregate Data
my_ldb_final = my_ldb_final.sort_values(by=['Year', 'Sub-Category', 'Mass/non-mass (subdivision)', 'Brand', 'Period'])
my_ldb_final.head(3)

,On/Offline,Year,Platform,Category,Sub-Category,Mass/non-mass (subdivision),Brand,Period,Sales Value,Sales Units
0,Offline,2023,Nielsen,SKINCARE,,Mass Medical,MEDIC MARKET,4,4253813.919,122416.178
1,Offline,2023,Nielsen,SKINCARE,,Mass Medical,MEDIC MARKET,5,3808601.820,101573.747
2,Offline,2023,Nielsen,SKINCARE,,Mass Medical,MEDIC MARKET,6,3955823.212,102667.598


In [25]:
# Get last month
last_month = datetime.now().replace(day=1) - timedelta(days=1)
month_abbr = last_month.strftime("%b").upper()
year = last_month.year

# Format the output as "MMM YYYY"
filemonth = last_month.strftime("%b %Y").upper()

# Print the result
print(f"{filemonth}")

MAY 2026


In [26]:
if not os.path.exists(f'../Generated Data/O+O/{filemonth}'):
        os.makedirs(f'../Generated Data/O+O/{filemonth}')

In [27]:
my_ldb_final.to_excel(f'../Generated Data/O+O/{filemonth}/MY LDB {filemonth} O+O.xlsx', index=False)